In [3]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [4]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Abdelfattah2022_Part2.h5ad")

In [5]:
adata = adata[adata.obs['donor_id'].isin(["ndGBM-01", "ndGBM-02", "ndGBM-03", "ndGBM-04", "ndGBM-05", "ndGBM-06", "ndGBM-07", "ndGBM-08", "ndGBM-09", "ndGBM-10", "ndGBM-11"])]

In [6]:
df_obs = pd.DataFrame(adata.obs)

In [7]:
del adata.obs

In [8]:
adata.raw.to_adata().X.A

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [9]:
adata = adata.raw.to_adata()

In [10]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [11]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [12]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [13]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [14]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF367,False,9726,4.780606,True,0.021459,0.012774,0.561512
SULT1B1,False,2137,1.050396,True,0.005889,0.004376,0.829614
TRIM63,False,257,0.126323,True,0.000825,0.000748,0.873097
HDHD2,False,43083,21.176523,True,0.109860,0.062678,0.566475
MORF4L2-AS1,False,2932,1.441162,True,0.006991,0.004746,0.765010
...,...,...,...,...,...,...,...
FAM237B,False,219,0.107645,True,0.000506,0.000370,0.856259
C13orf46,False,189,0.092899,True,0.000414,0.000272,0.759429
CARS1-AS1,False,111,0.054560,True,0.000212,0.000115,0.553398


In [15]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [16]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [17]:
adata.var = df_tmp

In [18]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [19]:
#adata.obs['author'] = df_obs['author']

In [20]:
adata.obs['donor_id'] = df_obs['donor_id']

In [21]:
metadata_data = {
    'Author': ['Abdelfattah2022', 'Abdelfattah2022', 'Abdelfattah2022', 'Abdelfattah2022', 'Abdelfattah2022',
               'Abdelfattah2022', 'Abdelfattah2022', 'Abdelfattah2022', 'Abdelfattah2022', 'Abdelfattah2022','Abdelfattah2022'],
    'donor_id': ["ndGBM-01", "ndGBM-02", "ndGBM-03", "ndGBM-04", "ndGBM-05", "ndGBM-06", "ndGBM-07", "ndGBM-08", "ndGBM-09",
                 "ndGBM-10", "ndGBM-11"],
    'stage': ["Primary", "Primary", "Primary", "Primary", "Primary", "Primary", "Primary", "Primary", "Primary", "Primary", "Primary"],
    'assay': ['10x 3\' v3'] * 11,
    'tissue': ["frontal lobe", "frontal lobe", "forebrain", "right parietal lobe", "right frontal lobe", "left frontal lobe", "basal ganglion", "forebrain", "brain", "left frontal lobe", "left temporal lobe"],
    'Cells': ['Total'] * 11,
    'Method': ['cell'] * 11
}

metadata_df = pd.DataFrame(metadata_data)

In [22]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

        donor_id           Author    stage      assay              tissue  \
0       ndGBM-01  Abdelfattah2022  Primary  10x 3' v3        frontal lobe   
1       ndGBM-01  Abdelfattah2022  Primary  10x 3' v3        frontal lobe   
2       ndGBM-01  Abdelfattah2022  Primary  10x 3' v3        frontal lobe   
3       ndGBM-01  Abdelfattah2022  Primary  10x 3' v3        frontal lobe   
4       ndGBM-01  Abdelfattah2022  Primary  10x 3' v3        frontal lobe   
...          ...              ...      ...        ...                 ...   
111284  ndGBM-11  Abdelfattah2022  Primary  10x 3' v3  left temporal lobe   
111285  ndGBM-11  Abdelfattah2022  Primary  10x 3' v3  left temporal lobe   
111286  ndGBM-11  Abdelfattah2022  Primary  10x 3' v3  left temporal lobe   
111287  ndGBM-11  Abdelfattah2022  Primary  10x 3' v3  left temporal lobe   
111288  ndGBM-11  Abdelfattah2022  Primary  10x 3' v3  left temporal lobe   

        Cells Method  
0       Total   cell  
1       Total   cell  
2     

In [25]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [26]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
ndGBM-01-A_AAACCTGAGAAGATTC-1-0-1,ndGBM-01,1217,1649.492065,Neoplastic,Differentiated-like,MES-like,Astrocyte,Neural Stem/Precursor Cells,malignant cell
ndGBM-01-A_AAACCTGAGACATAAC-1-0-1,ndGBM-01,2227,1636.858887,Neoplastic,Differentiated-like,AC-like,Oligodendrocyte,Neurons,malignant cell
ndGBM-01-A_AAACCTGAGGCACATG-1-0-1,ndGBM-01,869,1436.748291,Neoplastic,Differentiated-like,MES-like,Oligodendrocyte,Neurons,malignant cell
ndGBM-01-A_AAACCTGCAATCGAAA-1-0-1,ndGBM-01,2809,2174.696289,Neoplastic,Stem-like,OPC-like,Oligodendrocyte,Neurons,malignant cell
ndGBM-01-A_AAACCTGCACGGTAGA-1-0-1,ndGBM-01,1735,1866.502686,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
...,...,...,...,...,...,...,...,...,...
ndGBM-11-D_TTTGGTTCAAACAACA-1-0-1,ndGBM-11,773,1211.528442,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Macrophages,macrophage
ndGBM-11-D_TTTGGTTCACGCTTTC-1-0-1,ndGBM-11,1250,1520.987915,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Macrophages,microglial cell
ndGBM-11-D_TTTGTCAAGCGTGAGT-1-0-1,ndGBM-11,808,1344.932251,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Macrophages,microglial cell
ndGBM-11-D_TTTGTCAGTAGCTCCG-1-0-1,ndGBM-11,836,1289.346558,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Macrophages,macrophage


In [27]:
merged_obs_df.index= df_obs.index

In [28]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [29]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [30]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [31]:
del merged_obs_df['donor_id_y']

In [32]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [33]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [34]:
adata.obs = merged_obs_df

In [37]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    643 total control genes are used. (0:00:03)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    685 total control genes are used. (0:00:03)
-->     'phase', cell cycle phase (adata.obs)


In [38]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Abdelfattah2022_Part3.h5ad")